In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import shutil

In [2]:
DATA_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL")
PHOTO_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO")

MATCH_DIR_LABEL_REGEX = re.compile('TL_3.상추[0-9]+')
MATCH_DIR_PHOTO_REGEX = re.compile('TS_3.상추[0-9]+')

REF_PHOTO_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset")

MIN_DIR = 54
MAX_DIR = 75

In [3]:
list_of_labels = []
#Perform and exhuastive search, and yes this is intentional.
for entry in DATA_DIR.iterdir():
    if entry.is_dir() and MATCH_DIR_LABEL_REGEX.match(entry.name):
        dir_no = int(entry.name[-2] + entry.name[-1])
        if MIN_DIR <= dir_no and dir_no <= MAX_DIR:
            print(f"Adding NO:{dir_no}")
            list_of_labels.extend([subentry for subentry in entry.iterdir()])

Adding NO:71
Adding NO:70
Adding NO:64
Adding NO:63
Adding NO:55
Adding NO:54
Adding NO:62
Adding NO:65
Adding NO:72
Adding NO:75
Adding NO:74
Adding NO:73
Adding NO:60
Adding NO:67
Adding NO:58
Adding NO:56
Adding NO:69
Adding NO:57
Adding NO:68
Adding NO:66
Adding NO:59
Adding NO:61


In [4]:
list_of_labels[0:5]

[PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL/TL_3.상추71/C46_L01_03_006_649803.json'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL/TL_3.상추71/C46_L01_03_006_645089.json'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL/TL_3.상추71/C46_L01_03_006_649100.json'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL/TL_3.상추71/C46_L01_03_006_644732.json'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL/TL_3.상추71/C46_L01_03_006_644270.json')]

In [5]:
def safe_avg_of(data, column_name):
    column = [e[column_name] for e in data]
    if None in column:
        return None
    num_column = [float(n) for n in column]
    return sum(num_column) / len(num_column)

In [6]:
def growth_to_number(growth):
    if growth == '정식기':
        return 1
    elif growth == '생육기':
        return 2
    elif growth == '수확기':
        return 3
    raise ValueError("Some other value has been detected")

In [7]:
def parse_json_file(file_path):
    """Reads a single JSON file and returns a dictionary or None if failed."""
    try:
        # Using pathlib's read_text handles the file opening/closing for you
        #dir_no, file_path = file_info
        raw_data = file_path.read_text(encoding='utf-8')
        
        data = json.loads(raw_data)

        leaf_ids = []
        stem_ids = []

        for cat in data["categories"]:
            if cat["name"] == "잎":
                leaf_ids.append(cat["id"])
            else:
                stem_ids.append(cat["id"])

        environment = data["envrionments"]
        
        extracted = {
            #'dir_no': dir_no,
            'file_name': file_path.name.split(".")[0],
            'growth': growth_to_number(data["images"]["growth_stage"]),
            'captured': data["images"]["date_captured"],
            'kind': data["images"]["kind_type"],
            'temp': safe_avg_of(environment, "ti_value"),
            'humid': safe_avg_of(environment, "hi_value"),
            'co2': safe_avg_of(environment, "ci_value"),
            'light': safe_avg_of(environment, "ir_value"),
            'hyd_temp': safe_avg_of(environment, "tl_value"),
            'hyd_ec': safe_avg_of(environment, "ei_value"),
            'hyd_ph': safe_avg_of(environment, "pl_value"),
            'stem_area': sum([box["area"] if box["id"] in stem_ids else 0 for box in data["annotations"]]),
            'leaf_area': sum([box["area"] if box["id"] in leaf_ids else 0 for box in data["annotations"]]),
        }
        
        return extracted 
    except (json.JSONDecodeError, IOError) as e:
        print(f"Error parsing {file_path.name}: {e}")
    except TypeError as e:
        print(f"Error parsing {file_path.name}: {e}")
        return None

In [8]:
#test parse_json_file
parse_json_file(list_of_labels[0])

{'file_name': 'C46_L01_03_006_649803',
 'growth': 2,
 'captured': '2021-12-16 14:19:03',
 'kind': '로메인',
 'temp': 22.0,
 'humid': 50.0,
 'co2': 507.5,
 'light': 81.0,
 'hyd_temp': 21.3,
 'hyd_ec': 0.1,
 'hyd_ph': 6.3,
 'stem_area': 2870171,
 'leaf_area': 633812.83}

In [9]:
parsed_records = [parse_json_file(label_path) for label_path in list_of_labels]
df = pd.DataFrame(parsed_records)

In [10]:
df.head()

,file_name,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area
0,C46_L01_03_006_649803,2,2021-12-16 14:19:03,로메인,22.0,50.0,507.5,81.0,21.3,0.10,6.3,2870171.00,633812.83
1,C46_L01_03_006_645089,2,2021-11-23 12:19:03,로메인,18.0,23.0,458.5,83.0,18.5,0.08,6.6,241908.84,76982.72
2,C46_L01_03_006_649100,2,2021-12-12 04:19:03,로메인,21.0,39.0,493.5,85.0,19.9,0.10,6.4,3414952.27,244405.71
3,C46_L01_03_006_644732,1,2021-11-20 13:19:03,로메인,19.0,55.0,495.5,18.5,19.4,0.07,6.7,106801.97,71167.38
4,C46_L01_03_006_644270,1,2021-11-16 12:19:03,로메인,21.0,39.0,473.5,83.0,18.6,0.07,6.3,21401.00,14901.33


In [11]:
#Perform and exhuastive search, and move it to destination
for entry in PHOTO_DIR.iterdir():
    if entry.is_dir() and MATCH_DIR_PHOTO_REGEX.match(entry.name):
        dir_no = int(entry.name[-2] + entry.name[-1])
        if MIN_DIR <= dir_no and dir_no <= MAX_DIR:
            for subentry in entry.iterdir():
                if subentry.is_file():
                    #print(f"Moving {subentry} to {REF_PHOTO_DIR / subentry.name}")
                    shutil.copy(subentry, REF_PHOTO_DIR / subentry.name)

In [12]:
output_file = Path("plant_gotchi_dataset.parquet")

# We use 'pyarrow' as the engine and 'snappy' for fast compression
df.to_parquet(
    output_file, 
    engine='pyarrow', 
    compression='snappy',
    index=False  # Usually, we don't want the pandas row index in our data
)

print(f"Dataset successfully saved to: {output_file.absolute()}")

Dataset successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_dataset.parquet
